# Merkmalsnutzen je Modellfamilie: helfen Indizes anderen Modellen?

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** In der Datenreduktionsstudie (#155) brachten Vegetationsindizes der SVM nichts (0,848 statt 0,859). Ein Index ist meist ein **Verhältnis** zweier Bänder, z. B. NDVI. Die SVM vergleicht ganze Spektren über Abstände und braucht das Verhältnis nicht ausdrücklich. **Entscheidungsbäume** prüfen aber in jedem Split nur *ein* Merkmal, und **lineare Modelle** können kein Verhältnis bilden. Ihnen könnten Indizes helfen.

**Hypothese:** Für Extra Trees, HistGradientBoosting und logistische Regression verbessern Indizes die CV, und „nur Indizes + Meta“ liegt näher am vollen Spektrum als bei der SVM. Ein früher Hinweis: In der EDA-Ablation mit Random Forest lagen Indizes vor den Rohspektren (0,590 gegenüber 0,570).

**Vorgehen:** Vier Eingaben je Modell, kleines Raster, CV auf dem 80/20-Split (Experiment `band-reduction`). Die SVM-Werte kommen aus #155 (MLflow).

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Any

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression

from awp2.config import (
    LOGISTIC_REGRESSION_MAX_ITER,
    MLFLOW_REDUCTION_EXPERIMENT,
    MLFLOW_TRACKING_URI,
    MODEL_DOCS_FIGURES_DIR,
    REDUCTION_VARIANT_COL,
)
from awp2.experiment import TuneConfig
from awp2.models import (
    CombinedLabelClassifier,
    make_combined_extra_trees,
    make_combined_hist_gradient_boosting,
)
from awp2.plots import save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, ReductionVariant, reduction_curve, reduction_variants

base = PreprocessingConfig(use_spectral_standard_scale=True)
standard = {variant.name: variant for variant in reduction_variants(base)}
variants = [
    standard["all_bands"],
    ReductionVariant(
        name="plus_indices",
        family="all_bands",
        preprocessing=base.model_copy(update={"use_vegetation_indices": True}),
    ),
    standard["indices_only"],
    standard["sentinel2"],
]
models: dict[str, tuple[BaseEstimator, dict[str, list[Any]], bool]] = {
    "extra_trees": (
        make_combined_extra_trees(),
        {"estimator__max_features": [0.3, "sqrt"], "estimator__min_samples_leaf": [2]},
        False,
    ),
    "hist_gradient_boosting": (
        make_combined_hist_gradient_boosting(),
        {"estimator__learning_rate": [0.05, 0.1]},
        True,
    ),
    "logistic_regression": (
        CombinedLabelClassifier(
            LogisticRegression(class_weight="balanced", max_iter=LOGISTIC_REGRESSION_MAX_ITER)
        ),
        {"estimator__C": [1.0, 10.0, 100.0]},
        False,
    ),
}

## 1. Vier Eingaben je Modellfamilie

In [ ]:
curves = []
for name, (model, grid, balance) in models.items():
    curve = reduction_curve(
        model,
        variants,
        TuneConfig(
            name=f"family_{name}",
            approach="combined",
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"Merkmalsnutzen je Modellfamilie: {name}",
            param_grid=grid,
            preprocessing=base,
            balance_samples=balance,
        ),
    )
    curves.append(curve.assign(model=name))
results = pd.concat(curves, ignore_index=True)
results.pivot(index="model", columns=REDUCTION_VARIANT_COL, values=list(CURVE_METRICS)).round(3)

## 2. Vergleich mit der SVM (#155)

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
svm_runs = {
    "all_bands": "audit_svm_reference",
    "plus_indices": "audit_svm_indices",
    "indices_only": "audit_svm_indices_meta",
    "sentinel2": "reduction_svm_sentinel2",
}
svm_scores = {}
for variant, run_name in svm_runs.items():
    runs = mlflow.search_runs(
        experiment_names=[MLFLOW_REDUCTION_EXPERIMENT],
        filter_string=(
            f"attributes.run_name = '{run_name}' and params.`split.validation_size` = '0.2' "
            "and tags.`run.kind` = 'tuning'"
        ),
        order_by=["attributes.start_time DESC"],
    )
    svm_scores[variant] = runs["metrics.cv_bacc_combined_mean"].iloc[0]
combined = results.pivot(index="model", columns=REDUCTION_VARIANT_COL, values="bacc_combined")
combined.loc["svm"] = pd.Series(svm_scores)
order = [variant.name for variant in variants]
combined = combined[order]
gains = combined.sub(combined["all_bands"], axis=0).drop(columns="all_bands")
pd.concat({"CV-BAcc kombiniert": combined, "Differenz zu allen Bändern": gains}, axis=1).round(3)

In [ ]:
labels = {
    "all_bands": "alle Bänder",
    "plus_indices": "+ Indizes",
    "indices_only": "nur Indizes + Meta",
    "sentinel2": "Sentinel-2",
}
fig, ax = plt.subplots(figsize=(10, 4.5))
combined.rename(columns=labels).plot.bar(ax=ax, rot=0, width=0.8)
ax.set_ylabel("CV-BAcc kombiniert")
ax.set_title("Merkmalsnutzen je Modellfamilie")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize="small", loc="lower right")
save_doc_figure(fig, "indizes_modellfamilien", MODEL_DOCS_FIGURES_DIR)